In [1]:
import os
import sys
import yaml
from datetime import datetime

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from sklearn.preprocessing import StandardScaler

# 프로젝트 디렉토리 경로 설정
project_dir = "/home/youngjins/project/belief_trading"
os.chdir(project_dir)
# lib 폴더 경로 추가
sys.path.append(project_dir + "/lib")

# lib 폴더의 모듈 가져오기
from lib.model.hbt import (
    create_benchmark_model,
    create_proposed_model,
    HierarchicalModelTrainer,
)
from lib.utils.data import prepare_dataloaders
from lib.utils.confing import load_config
from lib.utils.train import train_model

# 설정 로드
config = load_config("train")

In [ ]:
# 소매 및 기관 투자자 데이터 로드
retail_path = os.path.join(
    project_dir, "data/binance/futures/um/dataset/retail_train.csv"
)
institutional_path = os.path.join(
    project_dir, "data/binance/futures/um/dataset/institutional_train.csv"
)

# 데이터 로드
retail_df = pd.read_csv(retail_path, index_col=0)
institutional_df = pd.read_csv(institutional_path, index_col=0)

# 데이터 기본 정보 확인
print(f"소매 투자자 데이터 크기: {retail_df.shape}")
print(f"기관 투자자 데이터 크기: {institutional_df.shape}")

# 타임스탬프 변환 (필요한 경우)
if "timestamp" in retail_df.columns:
    retail_df["timestamp"] = pd.to_datetime(retail_df["timestamp"])
    retail_df.set_index("timestamp", inplace=True)

if "timestamp" in institutional_df.columns:
    institutional_df["timestamp"] = pd.to_datetime(institutional_df["timestamp"])
    institutional_df.set_index("timestamp", inplace=True)

# 데이터 샘플 확인
print("소매 투자자 데이터 샘플:")
print(retail_df.head())
print("\n기관 투자자 데이터 샘플:")
print(institutional_df.head())

In [3]:
# 설정에서 필요한 파라미터 추출
context_length = config["model"][config["model"]["select"]]["context_length"]
device = config["model"]["training"]["device"]

data_sources = {"retail": retail_df, "institutional": institutional_df}

# 데이터로더 딕셔너리 생성
loaders = prepare_dataloaders(data_sources, config)

# 학습 데이터셋 통합
combined_train_dataset = ConcatDataset(
    [loaders["retail"]["train"].dataset, loaders["institutional"]["train"].dataset]
)
combined_train_loader = DataLoader(
    combined_train_dataset,
    batch_size=config["model"]["training"]["batch_size"],
    shuffle=True,
)

combined_val_dataset = ConcatDataset(
    [loaders["retail"]["val"].dataset, loaders["institutional"]["val"].dataset]
)
combined_val_loader = DataLoader(
    combined_val_dataset,
    batch_size=config["model"]["training"]["batch_size"],
    shuffle=True,
)

In [6]:
# 모델 파라미터 추출
model_config = config["model"]["hbt"]
ohlcv_dim = config["dataloader"]["ohlcv_dim"]
action_dim = model_config["action_dim"]
hidden_dim = model_config["hidden_dim"]
context_length = model_config["context_length"]
learning_rate = config["model"]["training"]["learning_rate"]

In [7]:
# 제안된 모델(use_other_player_beliefs=True) 생성
print("제안된 계층적 신념 트랜스포머(HBT) 모델 생성 중...")
proposed_model = create_proposed_model(
    ohlcv_dim=ohlcv_dim,
    action_dim=action_dim,
    hidden_dim=hidden_dim,
    context_length=context_length,
)

# 모델 트레이너 초기화
proposed_trainer = HierarchicalModelTrainer(model=proposed_model, lr=learning_rate)

# 제안된 모델 학습
print("제안된 모델 학습 시작...")
proposed_trainer, proposed_history = train_model(
    trainer=proposed_trainer,
    train_loader=combined_train_loader,
    val_loader=combined_val_loader,
    config=config,
)

# 모델 저장
config["model"]["save_path"] = model_config["save_path"]["proposed"]
proposed_trainer.save_model(config["model"]["save_path"])

print(
    f"모델이 성공적으로 학습되었으며 {config['model']['save_path']}에 저장되었습니다."
)

제안된 계층적 신념 트랜스포머(HBT) 모델 생성 중...
제안된 모델 학습 시작...


KeyError: 'save_path'

In [10]:
context_length

5